# 📝 Transcriptor de Audio y Video con IA (Whisper) — script1clic.com

**Funciona con GPU o CPU.** Con GPU (`Entorno de ejecución` → `Cambiar tipo de entorno` → `GPU T4`): transcribe mucho más rápido que con CPU.

**Pasos:** 1) Celda 1 (instalación, ~2-4 min) → 2) Celda 2 (interfaz).

> 🎬 **Funciona con audio Y video:** sube mp3, wav, m4a, ogg, flac, aac... o mp4, mov, mkv, webm. La herramienta extrae el audio automáticamente, no necesitas convertir nada primero.
> 🧠 **6 modelos para elegir, directo desde la interfaz** (sin re-ejecutar celdas): desde "Rápido" hasta "Máxima precisión", incluyendo variantes más nuevas y veloces que el Whisper clásico ("Turbo" y "Ultra rápido"). Cada opción muestra sus pros y contras al elegirla.
> 🔊 **Idioma automático o manual:** detecta el idioma solo, o elige uno entre casi 100 idiomas de la lista para forzarlo si el audio es corto/ruidoso.
> ❌ **Cancelar sin esperar:** si ya empezó a transcribir y subiste el archivo equivocado, el botón "Cancelar" la detiene al instante. Si el archivo TODAVÍA se está subiendo (barra de carga) y es el equivocado, recarga la pestaña (F5): es la única forma de cortar una subida en curso, por una limitación de Gradio.
> 📄 **Formatos de salida:** texto en párrafos (.txt), subtítulos (.srt / .vtt) y documento de Word (.docx).
> 💡 Si algo falla: vuelve a ejecutar la Celda 1 y luego la Celda 2.


In [ ]:
#@title 🔧 Celda 1: Instalación de dependencias (ejecutar una sola vez)
#@markdown Tarda 2-4 min. Espera el mensaje `✅ Todo listo`.

import sys, subprocess, warnings
warnings.filterwarnings("ignore")

# faster-whisper: transcripción rápida y precisa (basada en Whisper de OpenAI).
# Detecta el audio dentro de archivos de video automáticamente, sin pasos extra.
# python-docx: para poder exportar la transcripción como archivo de Word.
# ffmpeg ya viene instalado en Colab, pero lo reforzamos por si acaso.

print("📦 Instalando dependencias de script1clic.com...")
print("⏳ Esto puede tardar unos minutos, por favor espera...\n")

paquetes = [
    "faster-whisper",
    "gradio",
    "python-docx",
]

resultado = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U"] + paquetes,
    capture_output=True, text=True
)

subprocess.run(["apt-get", "-y", "-qq", "install", "ffmpeg"], check=False,
               capture_output=True, text=True)

if resultado.returncode != 0:
    print("⚠️ Ocurrió un problema durante la instalación.\n")
    print(resultado.stderr[-1500:])
    print("\n👉 Solución: vuelve a ejecutar esta misma Celda 1. Si el problema persiste, ve a 'Entorno de ejecución' → 'Reiniciar sesión' y vuelve a ejecutar la Celda 1.")
else:
    # Verificación en subproceso nuevo para evitar falsos positivos si
    # algo ya estaba cargado en esta sesión.
    check = subprocess.run(
        [sys.executable, "-c", "from faster_whisper import WhisperModel; import gradio; from docx import Document; print('OK')"],
        capture_output=True, text=True
    )
    if "OK" in check.stdout:
        print("✅ Todo listo. Las dependencias quedaron instaladas correctamente.")
        print("👉 Ahora ejecuta la Celda 2.")
    else:
        print("⚠️ Las dependencias se instalaron pero algo no importa correctamente.")
        print(check.stderr[-1500:])
        print("\n👉 Solución: ve a 'Entorno de ejecución' → 'Reiniciar sesión' y vuelve a ejecutar la Celda 1.")


In [ ]:
#@title 🚀 Celda 2: Iniciar Transcriptor (interfaz visual)
#@markdown El modelo y el idioma se eligen directamente **dentro de la interfaz** (más abajo, en el navegador), no aquí. Esta única opción sí conviene dejarla lista antes de abrir la interfaz:

#@markdown ### 🌐 Compartir enlace público (útil si quieres abrir la herramienta desde el celular)
compartir_publico = True #@param {type:"boolean"}

import os, traceback, tempfile, warnings
warnings.filterwarnings("ignore")

# Modelo que se precarga al abrir la interfaz, para que la primera
# transcripción no tenga que esperar la descarga. Se puede cambiar de
# modelo en cualquier momento desde el desplegable de la interfaz.
MODELO_INICIAL = "equilibrado"

MENSAJE_ERROR_AMIGABLE = """
⚠️ **Algo salió mal.** Suele deberse a: archivo dañado o en un formato no soportado, un archivo demasiado largo para la sesión gratuita de Colab, o memoria insuficiente para el modelo elegido (prueba uno más liviano).

👉 Prueba con otro archivo o con un modelo más rápido. Si persiste, re-ejecuta la Celda 1 y luego la Celda 2.
"""

MENSAJE_DEPENDENCIAS_FALTANTES = """
⚠️ **Faltan dependencias.** Ejecuta la Celda 1, espera `✅ Todo listo`, y luego la Celda 2.
"""

try:
    import torch
except Exception:
    torch = None

try:
    from faster_whisper import WhisperModel
    import gradio as gr
    from docx import Document
    dependencias_ok = True
except Exception:
    dependencias_ok = False
    print(MENSAJE_DEPENDENCIAS_FALTANTES)
    print("\n---- Detalle técnico ----")
    traceback.print_exc()

if dependencias_ok:
    try:
        gpu_disponible = torch is not None and torch.cuda.is_available()
        dispositivo = "cuda" if gpu_disponible else "cpu"
        compute_type = "float16" if gpu_disponible else "int8"
        print(f"🖥️ Usando: {dispositivo.upper()}")

        # ---------- Catálogo de modelos disponibles ----------
        # Todos usan la misma librería (faster-whisper), solo cambia el
        # nombre del modelo de Whisper que se descarga. "turbo" y
        # "ultra_rapido_ingles" son variantes más nuevas y más veloces que
        # el Whisper clásico, cada una con su propio trade-off.
        MODELOS_INFO = {
            "rapido": {
                "id": "tiny",
                "etiqueta": "⚡ Rápido",
                "descripcion": "⚡ El más veloz de todos. 👎 Se equivoca más, sobre todo con acentos fuertes o ruido de fondo.",
            },
            "equilibrado": {
                "id": "small",
                "etiqueta": "✅ Equilibrado (recomendado)",
                "descripcion": "✅ Buen balance velocidad/precisión. Recomendado para uso diario.",
            },
            "preciso": {
                "id": "medium",
                "etiqueta": "🎯 Preciso",
                "descripcion": "🎯 Más preciso que 'Equilibrado'. 👎 Tarda 2-3 veces más en procesar.",
            },
            "maxima_precision": {
                "id": "large-v3",
                "etiqueta": "🏆 Máxima precisión",
                "descripcion": "🏆 El más preciso, reconoce más de 99 idiomas. 👎 El más lento de todos.",
            },
            "turbo": {
                "id": "large-v3-turbo",
                "etiqueta": "🚀 Turbo (rápido, multilenguaje)",
                "descripcion": "🚀 Casi tan preciso como 'Máxima precisión' pero 6-8 veces más rápido. 👎 Algo peor si activas 'Traducir al inglés'.",
            },
            "ultra_rapido_ingles": {
                "id": "distil-large-v3.5",
                "etiqueta": "⚡⚡ Ultra rápido (solo inglés)",
                "descripcion": "⚡⚡ El más rápido de todos. 👎 Solo funciona bien con audio en INGLÉS: para español u otros idiomas, elige otro modelo.",
            },
        }

        # Lista casi completa de los ~99 idiomas que reconoce Whisper.
        # "auto" (detección automática) va primero; el resto en orden alfabético
        # para que sea fácil encontrar el idioma buscado.
        IDIOMAS_CHOICES = [
            ("🔍 Detectar automáticamente", "auto"),
            ("Afrikáans", "af"), ("Albanés", "sq"), ("Alemán", "de"), ("Amárico", "am"),
            ("Árabe", "ar"), ("Armenio", "hy"), ("Asamés", "as"), ("Azerbaiyano", "az"),
            ("Baskir", "ba"), ("Bengalí", "bn"), ("Bielorruso", "be"), ("Birmano", "my"),
            ("Bosnio", "bs"), ("Bretón", "br"), ("Búlgaro", "bg"), ("Cantonés", "yue"),
            ("Catalán", "ca"), ("Checo", "cs"), ("Chino", "zh"), ("Cingalés", "si"),
            ("Coreano", "ko"), ("Criollo haitiano", "ht"), ("Croata", "hr"), ("Danés", "da"),
            ("Eslovaco", "sk"), ("Esloveno", "sl"), ("Español", "es"), ("Estonio", "et"),
            ("Euskera (Vasco)", "eu"), ("Feroés", "fo"), ("Finlandés", "fi"), ("Francés", "fr"),
            ("Gallego", "gl"), ("Galés", "cy"), ("Georgiano", "ka"), ("Griego", "el"),
            ("Guyaratí", "gu"), ("Hausa", "ha"), ("Hawaiano", "haw"), ("Hebreo", "he"),
            ("Hindi", "hi"), ("Húngaro", "hu"), ("Indonesio", "id"), ("Inglés", "en"),
            ("Islandés", "is"), ("Italiano", "it"), ("Japonés", "ja"), ("Javanés", "jw"),
            ("Jemer", "km"), ("Kazajo", "kk"), ("Lao", "lo"), ("Latín", "la"),
            ("Letón", "lv"), ("Lingala", "ln"), ("Lituano", "lt"), ("Luxemburgués", "lb"),
            ("Macedonio", "mk"), ("Malayalam", "ml"), ("Malayo", "ms"), ("Malgache", "mg"),
            ("Maltés", "mt"), ("Maorí", "mi"), ("Maratí", "mr"), ("Mongol", "mn"),
            ("Neerlandés (Holandés)", "nl"), ("Nepalí", "ne"), ("Noruego", "no"),
            ("Noruego nynorsk", "nn"), ("Occitano", "oc"), ("Panyabí", "pa"), ("Pastú", "ps"),
            ("Persa", "fa"), ("Polaco", "pl"), ("Portugués", "pt"), ("Rumano", "ro"),
            ("Ruso", "ru"), ("Sánscrito", "sa"), ("Serbio", "sr"), ("Shona", "sn"),
            ("Sindhi", "sd"), ("Somalí", "so"), ("Suajili", "sw"), ("Sueco", "sv"),
            ("Sundanés", "su"), ("Tagalo", "tl"), ("Tailandés", "th"), ("Tamil", "ta"),
            ("Tártaro", "tt"), ("Tayiko", "tg"), ("Telugu", "te"), ("Tibetano", "bo"),
            ("Turco", "tr"), ("Turcomano", "tk"), ("Ucraniano", "uk"), ("Urdu", "ur"),
            ("Uzbeko", "uz"), ("Vietnamita", "vi"), ("Yidis", "yi"), ("Yoruba", "yo"),
        ]

        modelos_cargados = {}  # cache: clave de MODELOS_INFO -> instancia de WhisperModel ya cargada

        def obtener_modelo(clave_modelo):
            """Carga un modelo la primera vez que se pide, y lo reutiliza después
            (sin volver a descargarlo ni recargarlo) durante el resto de la sesión."""
            if clave_modelo not in modelos_cargados:
                id_modelo = MODELOS_INFO[clave_modelo]["id"]
                print(f"📥 Cargando modelo '{id_modelo}' por primera vez...")
                modelos_cargados[clave_modelo] = WhisperModel(id_modelo, device=dispositivo, compute_type=compute_type)
                print(f"✅ Modelo '{id_modelo}' listo (quedará en caché para la próxima vez).")
            return modelos_cargados[clave_modelo]

        # Precargamos el modelo por defecto para que la primera
        # transcripción del usuario no tenga que esperar la descarga.
        obtener_modelo(MODELO_INICIAL)
        print("✅ Modelo de transcripción listo.\n")

        # ---------- Utilidades de formato de tiempo y archivos ----------

        def formatear_duracion(segundos):
            """Convierte segundos a MM:SS o HH:MM:SS, para mostrar al usuario."""
            segundos = int(segundos or 0)
            h, resto = divmod(segundos, 3600)
            m, s = divmod(resto, 60)
            return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

        def formatear_srt_tiempo(segundos):
            ms_totales = int(round((segundos or 0) * 1000))
            h, resto = divmod(ms_totales, 3600000)
            m, resto = divmod(resto, 60000)
            s, ms = divmod(resto, 1000)
            return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

        def formatear_vtt_tiempo(segundos):
            return formatear_srt_tiempo(segundos).replace(",", ".")

        def guardar_archivo(contenido, sufijo):
            ruta = tempfile.mktemp(suffix=sufijo)
            with open(ruta, "w", encoding="utf-8") as f:
                f.write(contenido)
            return ruta

        # ---------- Párrafos, para lectura cómoda ----------

        def generar_parrafos(segmentos, con_marcas_tiempo, pausa_parrafo=2.5):
            """Agrupa los segmentos de Whisper en párrafos, cortando cuando
            hay una pausa larga (silencio) en el audio original."""
            parrafos, actual, inicio_parrafo, fin_anterior = [], [], None, None
            for seg in segmentos:
                texto = seg.text.strip()
                if not texto:
                    continue
                if fin_anterior is not None and (seg.start - fin_anterior) > pausa_parrafo and actual:
                    parrafos.append((inicio_parrafo, " ".join(actual)))
                    actual = []
                if not actual:
                    inicio_parrafo = seg.start
                actual.append(texto)
                fin_anterior = seg.end
            if actual:
                parrafos.append((inicio_parrafo, " ".join(actual)))

            if not parrafos:
                return "⚠️ No se detectó texto en el archivo."

            lineas = [f"[{formatear_duracion(inicio)}] {texto}" if con_marcas_tiempo else texto
                      for inicio, texto in parrafos]
            return "\n\n".join(lineas)

        # ---------- Subtítulos (SRT / VTT), agrupando palabra por palabra ----------
        # Usamos las marcas de tiempo por palabra (word_timestamps) para que
        # cada línea de subtítulo sea corta y fácil de leer, como en Netflix/YouTube.

        def agrupar_en_subtitulos(palabras, max_caracteres=42, max_palabras=10,
                                   max_duracion=6.0, pausa_corte=0.6):
            chunks, actual = [], []
            for inicio, fin, texto in palabras:
                if actual:
                    gap = inicio - actual[-1][1]
                    duracion_actual = actual[-1][1] - actual[0][0]
                    texto_actual = "".join(p[2] for p in actual)
                    if (gap > pausa_corte or len(actual) >= max_palabras
                            or len(texto_actual) + len(texto) > max_caracteres
                            or duracion_actual > max_duracion):
                        chunks.append(actual)
                        actual = []
                actual.append((inicio, fin, texto))
            if actual:
                chunks.append(actual)
            return chunks

        def generar_srt(palabras):
            if not palabras:
                return ""
            chunks = agrupar_en_subtitulos(palabras)
            bloques = []
            for i, chunk in enumerate(chunks, start=1):
                inicio, fin = chunk[0][0], chunk[-1][1]
                texto = "".join(p[2] for p in chunk).strip()
                bloques.append(f"{i}\n{formatear_srt_tiempo(inicio)} --> {formatear_srt_tiempo(fin)}\n{texto}\n")
            return "\n".join(bloques)

        def generar_vtt(palabras):
            if not palabras:
                return "WEBVTT\n\n"
            chunks = agrupar_en_subtitulos(palabras)
            lineas = ["WEBVTT", ""]
            for chunk in chunks:
                inicio, fin = chunk[0][0], chunk[-1][1]
                texto = "".join(p[2] for p in chunk).strip()
                lineas += [f"{formatear_vtt_tiempo(inicio)} --> {formatear_vtt_tiempo(fin)}", texto, ""]
            return "\n".join(lineas)

        # ---------- Word (.docx) ----------

        def generar_docx(texto_parrafos):
            doc = Document()
            doc.add_heading("Transcripción — script1clic.com", level=1)
            for parrafo in texto_parrafos.split("\n\n"):
                doc.add_paragraph(parrafo)
            ruta = tempfile.mktemp(suffix=".docx")
            doc.save(ruta)
            return ruta

        # ---------- Función principal que llama la interfaz ----------

        def transcribir_archivo(archivo, modelo_sel, idioma_sel, traducir_al_ingles, incluir_marcas_tiempo):
            sin_resultado = ("", None, None, None, None)  # texto, txt, srt, vtt, docx

            if archivo is None:
                yield "⚠️ Sube un audio o video primero.", *sin_resultado
                return

            if modelo_sel not in modelos_cargados:
                etiqueta = MODELOS_INFO[modelo_sel]["etiqueta"]
                yield f"📥 Cargando el modelo {etiqueta} por primera vez (1-2 min)... luego quedará listo al instante.", *sin_resultado

            try:
                modelo_activo = obtener_modelo(modelo_sel)
            except Exception:
                traceback.print_exc()
                yield MENSAJE_ERROR_AMIGABLE, *sin_resultado
                return

            yield "🔄 Transcribiendo... esto puede tardar según la duración del archivo. Puedes presionar 'Cancelar' en cualquier momento si subiste el archivo equivocado.", *sin_resultado

            try:
                # faster-whisper decodifica el audio con ffmpeg/PyAV por dentro,
                # así que funciona igual con un .mp3 que con un .mp4 o .mov:
                # simplemente toma la pista de audio del video automáticamente.
                idioma_param = None if idioma_sel == "auto" else idioma_sel
                tarea = "translate" if traducir_al_ingles else "transcribe"
                segmentos_gen, info = modelo_activo.transcribe(
                    archivo,
                    language=idioma_param,
                    task=tarea,
                    word_timestamps=True,
                    vad_filter=True,
                    beam_size=5,
                )
                segmentos = list(segmentos_gen)

                if not segmentos:
                    yield "⚠️ No se detectó voz en el archivo. Prueba con otro archivo.", *sin_resultado
                    return

                palabras = []
                for seg in segmentos:
                    if seg.words:
                        for w in seg.words:
                            palabras.append((w.start, w.end, w.word))

                texto_parrafos = generar_parrafos(segmentos, incluir_marcas_tiempo)
                srt_texto = generar_srt(palabras)
                vtt_texto = generar_vtt(palabras)

                ruta_txt = guardar_archivo(texto_parrafos, ".txt")
                ruta_srt = guardar_archivo(srt_texto, ".srt")
                ruta_vtt = guardar_archivo(vtt_texto, ".vtt")
                ruta_docx = generar_docx(texto_parrafos)

                idioma_detectado = getattr(info, "language", "?")
                prob = (getattr(info, "language_probability", 0) or 0) * 100
                duracion_txt = formatear_duracion(getattr(info, "duration", 0))
                nota_traduccion = " (traducido al inglés)" if traducir_al_ingles else ""
                resumen = (f"✅ ¡Listo!{nota_traduccion} Idioma detectado: {idioma_detectado} "
                           f"({prob:.0f}% confianza). Duración: {duracion_txt}.")

                yield resumen, texto_parrafos, ruta_txt, ruta_srt, ruta_vtt, ruta_docx

            except Exception:
                traceback.print_exc()
                yield MENSAJE_ERROR_AMIGABLE, *sin_resultado

        def cancelar_transcripcion():
            return "⏹️ Transcripción cancelada. Ya puedes subir otro archivo o cambiar las opciones.", "", None, None, None, None

        def describir_modelo(modelo_sel):
            return MODELOS_INFO[modelo_sel]["descripcion"]

        with gr.Blocks(title="Transcriptor de Audio y Video - script1clic.com", theme=gr.themes.Soft()) as interfaz:
            gr.Markdown("""
            # 📝 Transcriptor de Audio y Video con IA (Whisper)
            ### Powered by script1clic.com

            1. Sube un audio o video (cualquier formato: mp3, wav, m4a, mp4, mov, etc).
            2. Elige modelo e idioma (puedes cambiarlos cuando quieras, sin re-ejecutar celdas).
            3. Presiona "Transcribir" y descarga el resultado como texto, subtítulos (.srt / .vtt) o Word.
            4. ¿Ya empezó a transcribir y subiste el archivo equivocado? Presiona "Cancelar" y sube el correcto, sin esperar a que termine.
            5. ¿El archivo TODAVÍA se está subiendo (barra de carga) y es el equivocado? Recarga esta pestaña (F5) — ver aviso junto al botón de subir.
            """)

            with gr.Row():
                with gr.Column():
                    archivo_input = gr.File(label="🎵🎬 Sube tu audio o video (cualquier formato)", type="filepath")
                    gr.Markdown(
                        "⚠️ *Si el archivo es pesado y te equivocaste al subirlo: Gradio no deja cancelar "
                        "ni cambiar el archivo mientras se está subiendo (es una limitación de la plataforma, "
                        "no del código). La solución es **recargar esta pestaña (F5)** — no perderás el modelo "
                        "ya cargado ni necesitas volver a ejecutar celdas, solo se reinicia la página.*"
                    )

                    modelo_input = gr.Dropdown(
                        label="🧠 Modelo de transcripción",
                        choices=[(v["etiqueta"], k) for k, v in MODELOS_INFO.items()],
                        value=MODELO_INICIAL,
                    )
                    descripcion_modelo = gr.Markdown(MODELOS_INFO[MODELO_INICIAL]["descripcion"])
                    modelo_input.change(fn=describir_modelo, inputs=modelo_input, outputs=descripcion_modelo)

                    idioma_input = gr.Dropdown(label="🔊 Idioma del audio", choices=IDIOMAS_CHOICES, value="auto")
                    traducir_input = gr.Checkbox(label="🌍 Traducir el resultado al inglés (ignora el idioma de arriba)", value=False)
                    marcas_tiempo_input = gr.Checkbox(label="⏱️ Incluir marca de tiempo en cada párrafo del texto", value=False)

                    with gr.Row():
                        boton_transcribir = gr.Button("📝 Transcribir", variant="primary")
                        boton_cancelar = gr.Button("❌ Cancelar", variant="stop")
                    estado_output = gr.Textbox(label="📋 Estado", interactive=False)
                with gr.Column():
                    texto_output = gr.Textbox(label="📄 Texto transcrito (puedes editarlo)", lines=14, interactive=True)

            gr.Markdown("### ⬇️ Descargas")
            with gr.Row():
                archivo_txt = gr.File(label="📄 Texto (.txt)")
                archivo_srt = gr.File(label="🎬 Subtítulos (.srt)")
                archivo_vtt = gr.File(label="🌐 Subtítulos (.vtt)")
                archivo_docx = gr.File(label="📝 Word (.docx)")

            evento_transcripcion = boton_transcribir.click(
                fn=transcribir_archivo,
                inputs=[archivo_input, modelo_input, idioma_input, traducir_input, marcas_tiempo_input],
                outputs=[estado_output, texto_output, archivo_txt, archivo_srt, archivo_vtt, archivo_docx],
            )

            # El botón "Cancelar" interrumpe la transcripción en curso (evento de arriba)
            # al instante, para que el usuario no tenga que esperar a que termine si
            # subió el archivo equivocado. También limpia el estado y los resultados.
            boton_cancelar.click(
                fn=cancelar_transcripcion,
                inputs=None,
                outputs=[estado_output, texto_output, archivo_txt, archivo_srt, archivo_vtt, archivo_docx],
                cancels=[evento_transcripcion],
            )

            gr.Markdown("---\n*script1clic.com*")

        if not compartir_publico:
            print("ℹ️ Si quieres abrir esta herramienta desde tu celular, activa 'compartir_publico = True' arriba en esta misma celda y vuelve a ejecutarla.")

        interfaz.launch(share=compartir_publico, debug=False, inline=False)

    except Exception as e:
        print(MENSAJE_ERROR_AMIGABLE)
        print("\n---- Detalle técnico ----")
        traceback.print_exc()
